In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
from scipy.optimize import curve_fit
import pandas as pd
import h5py
from typing import Literal
import struct
from uncertainties import ufloat

plt.rcParams["figure.dpi"] = 100

cmap = plt.colormaps["viridis"]

In [ ]:
def plot_with_peaks(
    data,
    n_bins,
    divide=1.0,
    prominence=None,
    xlabel="",
    ylabel="Count",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
    label=None,
    scatter=False,
    offset_annotations_right=0.0,
    offset_annotations_up=None,
    color=None,
    fmt="--o",
    no_ebars=False,
    rng_annotations_up=0.0,  # what a terrible way to do this :/
    show_uncert=True
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        counts = np.array(counts)
        uncerts = np.sqrt(counts) / divide
        counts = counts / divide

        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        if scatter:
            plt.errorbar(
                bin_centers,
                counts,
                label=label,
                yerr=uncerts,
                fmt=fmt,
                markersize=4,
                capsize=0 if no_ebars else 3,
                elinewidth=0 if no_ebars else 1,
                c=color,
                linewidth=1,
                # ecolor="k",
                # mfc="white",
                # mec=color
            )
        else:
            plt.bar(
                bin_centers,
                counts,
                width=bin_width,
                align="center",
                bottom=bottom,
                label=label,
                color=color,
            )

        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.04f} ± {half_width:.04f} MeV" if show_uncert else f"{x:.03f} MeV",
                    (
                        x + offset_annotations_right,
                        (
                            y
                            + (
                                (height_scale * (0.01 * text_size))
                                if offset_annotations_up is None
                                else offset_annotations_up
                            ) + (np.random.random() - 0.5) * rng_annotations_up
                        ),
                    ),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    # plt.show()

In [ ]:
# thx chatgpt


def fwhm(E):
    return np.sqrt(
        (5e-4) ** 2 + 0.000004 * E  # electronics noise  # statistical broadening
    )


# Convert FWHM -> sigma
def sigma(E):
    return fwhm(E) / 2.355

# Loaders

In [ ]:
keys = [
    "Edep",
    "sourceE",
    "sourcePDG",
    "zInSoil",
    "rhoInSoil",
    # "sourceE"
]


def decode(b):
    return b.decode("utf-8")


decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["Measurement"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} interactions recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [ ]:
df15 = load_csv("finals/detection-15pc.csv")
df10 = load_csv("finals/detection-10pc.csv")
df5 = load_csv("finals/detection-5pc.csv")
df1 = load_csv("finals/detection-1pc.csv")

dfs = [df15, df10, df5, df1]
pcs = np.array([0.15, 0.10, 0.05, 0.01])
exposure_days = np.array([21.2131481481, 20.862037037, 20.462962963, 20.1990740741])

neutrons = lambda df: df[df["sourcePDG"] == 2112]

In [ ]:
# thx chatgpt

for df in [neutrons(df) for df in dfs]:
    range = df["Edep"].between(1.0, 10.0)
    # Smear each pulse energy
    smeared = np.random.normal(loc=df[range]["Edep"], scale=sigma(df[range]["Edep"]))
    plot_with_peaks(smeared[(smeared > 4.4) & (smeared < 4.46)], 128)
    plt.show()

# Analysis

## Masking Utilities

## Energy Spectrum

In [ ]:
def quadratic_model(x, a, b, c):
    return a * np.pow(x, 2) + b * x + c


def linear_model(x, m, c):
    return m * x + c


def analyse_peak_trend(
    e_low,
    e_high,
    label,
    mapper=lambda _: _,
    color=None,
    fmt=".",
    trendline="--",
    model=None,
    ax=None,
):

    ax = plt.gca() if ax is None else ax

    model = linear_model if model is None else model
    print(label)

    counts_per_day = []
    for df in dfs:
        counts_per_day.append(mapper(df)["Edep"].between(e_low, e_high).sum())

    uncert = [np.sqrt(n) for n in counts_per_day]
    counts_per_day /= exposure_days
    uncert /= exposure_days

    ax.errorbar(
        pcs,
        counts_per_day,
        yerr=uncert,
        fmt=fmt,
        label=label,
        c=color,
        elinewidth=1,
        capsize=2,
        markersize=5,
        mfc="white",
        ecolor="k",
    )
    # ax.set_title(f"Detector Counts per Day vs. Soil Organic Carbon Content")
    # ax.set_xlabel("Soil Organic Carbon Volume Fraction")
    # ax.set_ylabel("Detector Counts per Day")

    opt, cov = curve_fit(
        model,
        pcs,
        counts_per_day,
        sigma=uncert,
        absolute_sigma=True,
    )
    # gradient, gradient_u = opt[0], np.sqrt(cov[0, 0])
    x = np.linspace(pcs.min(), pcs.max(), 64)
    ax.plot(x, model(x, *opt), trendline, c=color)

    resids = counts_per_day - model(pcs, *opt)
    chi_squared = np.sum((resids / uncert) ** 2)
    dof = len(pcs) - len(opt)
    reduced_chi_squared = chi_squared / dof
    # print(f"  {reduced_chi_squared=:.2f}")
    # for i, param in enumerate(opt):
    # print(f"  opt[{i}]: {param:.0f} +/- {np.sqrt(cov[i, i]):.0f}")

    m = opt[0]
    m_u = np.sqrt(cov[0, 0])
    b = opt[1]
    b_u = np.sqrt(cov[1, 1])
    uncert_1day = np.sqrt(max(counts_per_day))
    sens = 100 * uncert_1day / ufloat(m, m_u)
    print(f"  ${m:.0f} \\pm {m_u:.0f}$ & {reduced_chi_squared:.2f}")
    print(f"  z={m/m_u}")
    print(f"  sens=${sens.n:.3f} \\pm {sens.s:.3f}$")


analyse_peak_trend(4.4375, 4.4385, "4.438 MeV", color="tab:red", fmt="o")
analyse_peak_trend(
    4.4375,
    4.4385,
    None,
    color="tab:red",
    fmt="o",
    mapper=neutrons,
    trendline=":",
)

analyse_peak_trend(4.4385, 4.4395, "4.439 MeV", color="tab:orange", fmt="s")
analyse_peak_trend(
    4.4385,
    4.4395,
    None,
    color="tab:orange",
    fmt="s",
    mapper=neutrons,
    trendline=":",
)

analyse_peak_trend(4.4375, 4.4395, "4.438 & 4.439 MeV", color="k", fmt=".")
analyse_peak_trend(
    4.4375,
    4.4395,
    None,
    color="k",
    fmt=".",
    mapper=neutrons,
    trendline=":",
)

analyse_peak_trend(4.422, 4.424, "4.423 MeV", color="tab:green", fmt="^")
analyse_peak_trend(
    4.422,
    4.424,
    None,
    color="tab:green",
    fmt="^",
    mapper=neutrons,
    trendline=":",
)

analyse_peak_trend(4.404, 4.408, "4.406 MeV", color="tab:blue", fmt="x")
analyse_peak_trend(
    4.404,
    4.408,
    None,
    color="tab:blue",
    fmt="x",
    mapper=neutrons,
    trendline=":",
)
# analyse_peak_trend(4.933, 4.935, "4.934 MeV", color="tab:blue", fmt="^")

# analyse_peak_trend(
#     4.933,
#     4.935,
#     None,
#     color="tab:blue",
#     fmt="^",
#     mapper=neutrons,
#     trendline="-.",
# )
plt.plot([], [], "--", label="All Particles", color="gray")
plt.plot([], [], ":", label="Neutrons Only", color="gray")

# analyse_peak_trend(6.128, 6.130)
# analyse_peak_trend(1.367, 1.369)
# analyse_peak_trend(5, 10)
plt.ylim(top=100)
plt.legend(fontsize=8)
# plt.yscale("log")
plt.show()

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

analyse_peak_trend(0.510, 0.512, "All Particles", color="tab:red", fmt="o", ax=ax1)
analyse_peak_trend(
    0.510,
    0.512,
    "Neutrons Only",
    color="tab:blue",
    fmt="^",
    mapper=neutrons,
    ax=ax2
)
# code taken from https://matplotlib.org/stable/gallery/subplots_axes_and_figures/broken_axis.html
ax1.set_ylim(43700, 47000)  # outliers only
ax2.set_ylim(1450, 1670)  # most of the data
ax1.spines.bottom.set_visible(False)
ax2.spines.top.set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # don't put tick labels at the top
ax2.xaxis.tick_bottom()

# ax1.errorbar([], [], fmt="o", c="tab:red", label="0.511 MeV (All)")
# ax1.errorbar([], [], fmt="^", c="tab:blue", label="0.511 MeV (Neutrons Only)")

d = .5  # proportion of vertical to horizontal extent of the slanted line
kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)
ax1.plot([0, 1], [0, 0], transform=ax1.transAxes, **kwargs)
ax2.plot([0, 1], [1, 1], transform=ax2.transAxes, **kwargs)

# analyse_peak_trend(1.778, 1.780, "1.779 MeV (All)")
# analyse_peak_trend(1.778, 1.780, "1.779 MeV (Neutrons)", mapper=neutrons)

# analyse_peak_trend(2.222, 2.224, "2.223 MeV (All)")
# analyse_peak_trend(2.222, 2.224, "2.223 MeV (Neutrons)", mapper=neutrons)

# analyse_peak_trend(7.134, 7.136, "7.135 MeV (All)")
# analyse_peak_trend(7.134, 7.136, "7.135 MeV (Neutrons)", mapper=neutrons)

# analyse_peak_trend(7.644, 7.646, "7.645 MeV (All)")
# analyse_peak_trend(7.644, 7.646, "7.645 MeV (Neutrons)", mapper=neutrons)


# analyse_peak_trend(0.05, 0.2, "Low energy (All)", fmt="x")
# analyse_peak_trend(3.538, 3.540, "3.539 MeV (All)", fmt="s")
# analyse_peak_trend(0.05, 0.2, "Low Energy (Neutrons)", fmt="x", mapper=neutrons)
# analyse_peak_trend(2.753, 2.755, "2.754 MeV (All)", fmt="s")
# analyse_peak_trend(3.538, 3.540, "3.539 MeV (Neutron)", fmt="s", mapper=neutrons)
# analyse_peak_trend(2.753, 2.755, "2.754 MeV (Neutron)", fmt="s", mapper=neutrons)

ax1.legend()
ax2.legend()
fig.supxlabel("Soil Organic Carbon Volume Fraction")
fig.supylabel("Detector Counts per Day")
plt.suptitle(f"Detector Counts per Day vs. Soil Organic Carbon Content,\nfor 0.511 MeV Gamma Ray Peak")
plt.show()

In [ ]:
def get_peak_ratios(energies_a, energies_b, n_bins=100, prominence=100):

    counts_a, bins_a = np.histogram(energies_a, bins=n_bins, density=False)
    counts_b, bins_b = np.histogram(energies_b, bins=n_bins, density=False)

    bin_centers_a = (bins_a[:-1] + bins_a[1:]) / 2.0
    bin_centers_b = (bins_b[:-1] + bins_b[1:]) / 2.0

    peaks_a, _ = find_peaks(counts_a, prominence=prominence)
    peaks_b, _ = find_peaks(counts_b, prominence=prominence)

    ratios = []
    for idx_a, e_a in enumerate(bin_centers_a[peaks_a]):
        if any(abs(e_a - e_b) < 0.01 for e_b in bin_centers_b[peaks_b]):
            idx_b = np.argmin([abs(e_a - e_b) for e_b in bin_centers_b[peaks_b]])
            ratio = counts_a[peaks_a][idx_a] / counts_b[peaks_b][idx_b]
            ratios.append(ratio)

    return np.array(ratios)


for df, pc in zip(dfs, pcs):
    mev_2_to_10 = df["Edep"].between(2.0, 10.0)
    ratios = get_peak_ratios(
        df[mev_2_to_10]["Edep"],
        df[mev_2_to_10 & (df["sourcePDG"] == 2112)]["Edep"],
        n_bins=4096,
        prominence=150,
    )

    print(pd.DataFrame(ratios, columns=[str(pc)]).agg(["mean", "std", "sem", "count"]))

In [ ]:
def plot_energy_spectrum(
    df,
    e_low,
    e_high,
    n_bins=512,
    prominence=None,
    clip=None,
    text_size=9,
    label=None,
    divide=1.0,
    color=None,
    scatter=True,
    fmt="o--",
    no_ebars=False,
    offset_annotations_up=0,
    offset_annotations_right=0.001,
    rng_annotations_up=0.0,
    show_uncert=True,
):
    energies = df[df["Edep"].between(e_low, e_high)]["Edep"]
    plot_with_peaks(
        data=energies,
        n_bins=n_bins,
        prominence=prominence,
        xlabel="Energy Deposited (MeV)",
        title="Germanium Detector Counts per Day",
        clip_counts=clip,
        text_size=text_size,
        label=label,
        scatter=scatter,
        divide=divide,
        ylabel="Counts Per Day" if divide != 1.0 else "Counts",
        offset_annotations_right=offset_annotations_right,
        offset_annotations_up=offset_annotations_up,
        color=color,
        fmt=fmt,
        no_ebars=no_ebars,
        rng_annotations_up=rng_annotations_up,
        show_uncert=show_uncert,
    )


plt.rcParams["figure.dpi"] = 512

plot_energy_spectrum(
    df15,
    0,
    1,
    n_bins=2048,
    prominence=1000,
    scatter=False,
    divide=exposure_days[0],
    clip=2000,
)

plt.show()
# plt.rcParams["figure.dpi"] = 100

for pc, df, exposure in zip(pcs, dfs, exposure_days):
    plot_energy_spectrum(
        df,
        4.4,
        4.462,
        n_bins=256,
        prominence=500,
        divide=exposure,
        color=cmap(pc * 6),
        fmt=":",
        no_ebars=True,
        show_uncert=False,
        offset_annotations_right=0.001
    )

for pc, df, exposure, fmt, prom in zip(
    pcs, dfs, exposure_days, ["o", "s", "^", "x"], [5, 10000, 10000, 10000]
):
    plot_energy_spectrum(
        df,
        4.40,
        4.462,
        n_bins=256,
        prominence=prom,
        divide=exposure,
        label=f"{pc:.0%}",
        color=cmap(pc * 6),
        fmt=fmt,
        show_uncert=False,
        offset_annotations_right=0.001
    )

plt.legend()
plt.show()

for pc, df, exposure in zip(pcs, dfs, exposure_days):
    plot_energy_spectrum(
        df,
        4.0,
        5.0,
        n_bins=256,
        prominence=10000,
        label=f"{pc:.0%}",
        divide=exposure,
        color=cmap(pc * 6),
        fmt="o",
    )
    # plot_energy_spectrum(df, 7.0, 7.2, n_bins=256, prominence=100, label=f"{pc}")
    # plot_energy_spectrum(df, 5.5, 6.3, n_bins=1024, prominence=100, divide=exposure, label=f"{pc}")

plt.legend()
plt.show()

for pc, df, exposure in zip(pcs, dfs, exposure_days):
    plot_energy_spectrum(
        df,
        5.5,
        6.3,
        n_bins=1024,
        prominence=10000,
        divide=exposure,
        label=f"{pc:.0%}",
        color=cmap(pc * 6),
    )
    # plot_energy_spectrum(df, 7.0, 7.2, n_bins=256, prominence=100, label=f"{pc}")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.yscale("log")
plot_energy_spectrum(
    df15,
    2,
    10,
    n_bins=2048,
    prominence=50,
    scatter=False,
    divide=exposure_days[0],
    text_size=5,
    offset_annotations_up=0,
    offset_annotations_right=0.05,
    rng_annotations_up=50
)

plt.show()

In [ ]:
# plt.figure(figsize=(5, 7))

plt.title("Detection of Particles at Detector,\nby Position of Source in Soil")

plt.hexbin(
    df["rhoInSoil"] / 10,
    df["zInSoil"] / 10,
    # C=df["Edep"],
    # reduce_C_function=np.mean,
    linewidths=0.4,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.xlabel("Radius from Origin, $\\rho$ (cm)")
plt.ylabel("Depth, $z$ (cm)")
plt.colorbar(label="Count")

plt.tight_layout()
plt.show()

In [ ]:
for i, (df, pc) in enumerate(zip(dfs, pcs)):
    plt.title(f"Energy Deposited in Detector vs. Energy at Production\n({pc:.0%} Soil Carbon)")

    mask = df["sourceE"].between(0.0, 5.0)
    plt.hexbin(
        df[mask]["sourceE"],
        df[mask]["Edep"],
        # C=df["Edep"],
        # reduce_C_function=np.sum,
        linewidths=0.5,
        # vmax=100,
        norm=LogNorm(),
    )
    plt.colorbar(label="Count")
    plt.xlabel("Gamma Ray Energy at Production (MeV)")
    plt.ylabel("Energy Deposited in Detector (MeV)")
    plt.xlim(0.0, 5.0)
    plt.ylim(0, 5.0)
    
    plt.show()
    
for i, (df, pc) in enumerate(zip([neutrons(df) for df in dfs], pcs)):
    plt.title(f"Energy Deposited in Detector vs. Energy at Production\n({pc:.0%} Soil Carbon, Neutrons Only)")

    mask = df["sourceE"].between(0.0, 5.0)
    plt.hexbin(
        df[mask]["sourceE"],
        df[mask]["Edep"],
        # C=df["Edep"],
        # reduce_C_function=np.sum,
        linewidths=0.5,
        # vmax=100,
        norm=LogNorm(),
    )
    plt.colorbar(label="Count")
    plt.xlabel("Gamma Ray Energy at Production (MeV)")
    plt.ylabel("Energy Deposited in Detector (MeV)")
    plt.xlim(0.0, 5.0)
    plt.ylim(0, 5.0)
    
    plt.show()